Copyright Matlantis Corp. as contributors to Matlantis contrib project

# Steered MD (SMD) シミュレーション

PLUMEDの`MOVINGRESTRAINT`機能を使用して、Cu表面原子をz方向に徐々に引き離すSteered MDを実行します。
これにより、反応座標空間（z = 10.2 Å → 18.0 Å）を連続的に探索するトラジェクトリを取得し、次のステップでアンブレラサンプリングの初期構造を抽出するために使用します。

## Step 1: PLUMED環境の設定
Steered MDを行う際に使用するPLUMEDは外部ライブラリであるため、Pythonからカーネルを正しく呼び出せるようにパスと環境変数を設定します。

**note**: PLUMEDのインストール方法については [00_README_ja.ipynb](./00_README_ja.ipynb) を参照してください。インストール先に合わせて `PLUMED_ROOT` のパスを適宜修正してください。

In [ ]:
# PLUMED environment

import os
import sys

# PLUMEDへのパス設定
PLUMED_ROOT   = os.path.expanduser("~/local/plumed-2.9.0")  # 必要に応じてplumedをインストールしたdirに修正する
plumed_bin    = os.path.join(PLUMED_ROOT, "bin")
plumed_lib    = os.path.join(PLUMED_ROOT, "lib")
plumed_kernel = os.path.join(plumed_lib, "libplumedKernel.so")

# 環境変数の設定
os.environ["PATH"]            = f"{plumed_bin}:{os.environ.get('PATH', '')}"
os.environ["LD_LIBRARY_PATH"] = f"{plumed_lib}:{os.environ.get('LD_LIBRARY_PATH', '')}"
os.environ["PLUMED_KERNEL"]   = str(plumed_kernel)

# Pythonライブラリパスの追加
if str(PLUMED_ROOT) not in sys.path:
    sys.path.append(str(PLUMED_ROOT))

## Step 2: ライブラリのインポートとCalculatorの設定

MDシミュレーションに必要なASEモジュール、PLUMEDラッパー、PFP Calculatorをインポートし、PFPの計算条件を設定します。

In [ ]:
import numpy as np

# ASE
from ase import units
from ase.io import read, write
from ase.constraints import FixAtoms
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution, Stationary
from ase.calculators.plumed import Plumed

# PFP
from pfp_api_client.pfp.calculators.ase_calculator import ASECalculator
from pfp_api_client.pfp.estimator import Estimator

from pfcc_extras import view_ngl

calc_mode = "PBE_PLUS_D3"
method_type = "PFVM_D3_PFVM"
model_version = "v8.0.0"
estimator = Estimator(calc_mode=calc_mode, method_type=method_type, model_version=model_version)
calculator = ASECalculator(estimator)

## Step 3: 初期構造の読み込み

NPzT平衡化MDで作成された構造を読み込み、Cuスラブの下層（z < 4.0 Å）を固定します。

**note**: `output/02_md_equilibrium/` にファイルがない場合は、`assets/02_equilibrium_npzt_md/mdtraj_eq.xyz` から読み込むことで、このノートブックを独立して実行できます。

In [ ]:
# outputから読み込み、なければassetsから読み込む
inp_file = './output/02_md_equilibrium/mdtraj_eq.xyz'
if not os.path.exists(inp_file):
    inp_file = './assets/02_equilibrium_npzt_md/mdtraj_eq.xyz'
    print(f"outputにファイルが見つからないため、assetsから読み込みます: {inp_file}")

atoms = read(inp_file)
atoms.wrap()
v = view_ngl(atoms, representations="ball+stick")
display(v)

In [ ]:
# Cuの1, 2層目を固定する

thresh = 4.0
constraint = FixAtoms(mask=atoms.positions[:, 2] < thresh)
atoms.set_constraint(constraint)
constraint

## Step 4: Steered MDの実行 (PLUMED)

PLUMEDの`MOVINGRESTRAINT`を使って、268番目の原子（Cu表面原子、ASE index: 267）のz座標を10.2 Åから18.0 Åまで徐々に移動させます。

**PLUMEDの設定ポイント:**
* `UNITS LENGTH=A ENERGY=eV`: 単位系をÅ、eVに指定。
* `POSITION ATOM=268`: 束縛対象の原子座標を取得（PLUMEDは**1始まり**のindexを使用）。
* `MOVINGRESTRAINT`: 調和ポテンシャルの中心を`AT0`から`AT1`まで線形に移動。

| パラメータ | 値 | 説明 |
|:---|:---|:---|
| バネ定数 (KAPPA) | 10.0 eV/Å² | SMDの拘束力 |
| 開始位置 (AT0) | 10.20 Å | 反応座標の開始値 |
| 終了位置 (AT1) | 18.00 Å | 反応座標の終了値 |
| ステップ数 | 200,000 (= 200 ps) | |
| 温度 | 375 K | Langevin thermostat |
| 摩擦係数 | 0.002 /fs | |

In [ ]:
atoms.get_positions()[267]

In [ ]:
out_dir = f"./output/03_steered_md_moving_restrain/"
os.makedirs(out_dir, exist_ok=True)

# ------------------------------------------------------------
# PLUMED Settings　of Collective Variables
# ------------------------------------------------------------
cv_start = 10.2
cv_end   = 18.0
total_steps = 200_000

plumed_setting = [
    f"UNITS LENGTH=A ENERGY=eV",
    f"t: TIME",

    # 1. 268番目のCuの原子座標を取得
    f"pos: POSITION ATOM=268",

    # 2. SMDの設定
    f"restraint: MOVINGRESTRAINT ARG=pos.z "
    f"STEP0=0             AT0={cv_start:.2f} KAPPA0=10.0 "
    f"STEP1={total_steps} AT1={cv_end:.2f}   KAPPA1=10.0",

    # 3. 出力の設定
    f"PRINT STRIDE=100 ARG=pos.z,restraint.bias,restraint.force2,restraint.work FILE={out_dir}/COLVAR_SMD",
    f"FLUSH STRIDE=100",
]
print(plumed_setting)

# ------------------------------------------------------------
# Molecular Dynamics
# ------------------------------------------------------------
timestep = 1 * units.fs
ps = 1000 * units.fs
temperature = 375

# PLUMED
atoms.calc = Plumed(calc=calculator, input=plumed_setting, timestep=timestep, atoms=atoms, kT=1)

# Set the momenta corresponding to the given "temperature"
MaxwellBoltzmannDistribution(atoms, temperature_K=temperature,force_temp=True)
Stationary(atoms)  # Set zero total momentum to avoid drifting

# Dynamics
dyn = Langevin(atoms, 
               timestep, 
               temperature_K=temperature, 
               friction=0.002/units.fs, 
               trajectory=f'{out_dir}/md-dyn.traj', 
               logfile=f'{out_dir}/md-dyn.log', 
               loginterval=100)

dyn.run(total_steps)

write(f'{out_dir}/md-smd-restart.xyz', atoms)

## Next Step
これで、Steered MDシミュレーションが完了しました。
次の [04_select_umbrella_sampling_initial_structures_ja.ipynb](./04_select_umbrella_sampling_initial_structures_ja.ipynb) では、このSMDトラジェクトリから各アンブレラウィンドウの初期構造を抽出します。